<a href="https://colab.research.google.com/github/takuonakashima/ai-security-workshop/blob/main/MNIST_simple_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# データの変換ルールとダウンロード、ベルトコンベア(train_loader)の作成
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)


# 1. 画像分類AIの構造（設計図）を定義する
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1) # 画像の特徴を抽出する層
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)     # 特徴から数字を判定する層
        self.fc2 = nn.Linear(128, 10)       # 0〜9の10クラスに分類

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x # 0〜9のそれぞれの数字っぽさの「生のスコア（Logit）」を出力

# 2. モデルの実体を作成し、GPU環境にセットする
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

# =====================================================================
# 学習済みの「脳（パラメータ）」をダウンロードして読み込む
import os
import urllib.request

print("GitHubから学習済みモデルをダウンロードしています...")
url = 'https://raw.githubusercontent.com/takuonakashima/ai-security-workshop/main/mnist_cnn.pth'
urllib.request.urlretrieve(url, 'mnist_cnn.pth')

# ダウンロードした脳みそをAIに注入する
model.load_state_dict(torch.load('mnist_cnn.pth', map_location=device))
# =====================================================================

# 3. AIモデルを「推論（テスト）モード」に切り替える
model.eval()

# データローダーから画像と正解ラベルを1つ取り出す
dataiter = iter(train_loader)
images, labels = next(dataiter)

# 4. 用意した画像をGPUに送り、AIに入力する
images = images.to(device)
labels = labels.to(device)

# AIに予測させる（推論の実行）
output = model(images)

# 5. AIの出力（生のスコア）を「確率（0〜100%）」に変換する
probabilities = F.softmax(output, dim=1)

# 最も確率が高い数字（AIの最終決定）と、その確率を取得する
max_prob, predicted_class = torch.max(probabilities, 1)

# 6. 結果の表示
print("=== 🛡️ 攻撃前のAIの推論結果 ===")
print(f"AIの予測: 『 {predicted_class.item()} 』")
print(f"確信度（確率）: {max_prob.item() * 100:.2f} %")
print(f"本当の正解: 『 {labels[0].item()} 』")

if predicted_class.item() == labels[0].item():
    print("\n✅ AIは正確に画像を認識しています！\nこの優秀なAIに対して、午後から『見えないノイズ』を使って騙す攻撃（回避攻撃）を仕掛けます。")
else:
    print("\n❌ AIが間違えました。（※乱数の運で間違えることがあります。セルを再実行して別の画像で試してください）")

100%|██████████| 9.91M/9.91M [00:00<00:00, 40.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.15MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.12MB/s]


GitHubから学習済みモデルをダウンロードしています...
=== 🛡️ 攻撃前のAIの推論結果 ===
AIの予測: 『 8 』
確信度（確率）: 100.00 %
本当の正解: 『 8 』

✅ AIは正確に画像を認識しています！
この優秀なAIに対して、午後から『見えないノイズ』を使って騙す攻撃（回避攻撃）を仕掛けます。
